# 03 Date dimension


In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes requests papermill
import os,sys,json,pandas as pd
from google.cloud import bigquery
PROJECT_ID="your-gcp-project-id"  # замініть у всіх ноутбуках
LOCATION="EU"; creds=None
if "google.colab" in sys.modules:
 from google.colab import auth; auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
 from google.oauth2 import service_account
 creds=service_account.Credentials.from_service_account_info(json.loads(os.environ["GCP_SA_KEY"]),scopes=["https://www.googleapis.com/auth/cloud-platform"])
client=bigquery.Client(project=PROJECT_ID,location=LOCATION,credentials=creds)
print("проєкт:",client.project)

In [ ]:
R=f"{PROJECT_ID}.nbu_raw.raw_rates"; T=f"{PROJECT_ID}.nbu_dwh.dim_date"; b=client.query(f"SELECT MIN(business_date) mi,MAX(business_date) ma FROM `{R}`").to_dataframe().iloc[0]; assert pd.notna(b.mi)
x=pd.DataFrame({"full_date":pd.date_range(f"{pd.Timestamp(b.mi).year}-01-01",f"{pd.Timestamp(b.ma).year}-12-31")}); x["date_key"]=x.full_date.dt.strftime("%Y%m%d").astype(int); x["year"]=x.full_date.dt.year; x["quarter"]=x.full_date.dt.quarter; x["month"]=x.full_date.dt.month; x["year_month"]=x.full_date.dt.strftime("%Y-%m"); x["day_name"]=x.full_date.dt.day_name(); x["is_weekend"]=x.full_date.dt.dayofweek>=5; x.full_date=x.full_date.dt.date; x=x[["date_key","full_date","year","quarter","month","year_month","day_name","is_weekend"]]; u=pd.DataFrame([[-1,pd.Timestamp("1900-01-01").date(),-1,-1,-1,"Unknown","Unknown",False]],columns=x.columns); x=pd.concat([u,x],ignore_index=True)
s=[bigquery.SchemaField("date_key","INTEGER"),bigquery.SchemaField("full_date","DATE"),bigquery.SchemaField("year","INTEGER"),bigquery.SchemaField("quarter","INTEGER"),bigquery.SchemaField("month","INTEGER"),bigquery.SchemaField("year_month","STRING"),bigquery.SchemaField("day_name","STRING"),bigquery.SchemaField("is_weekend","BOOLEAN")]; client.load_table_from_dataframe(x,T,job_config=bigquery.LoadJobConfig(schema=s,write_disposition="WRITE_TRUNCATE")).result(); k=pd.to_datetime(x.loc[x.date_key!=-1,"full_date"]); assert k.diff().dropna().eq(pd.Timedelta(days=1)).all() and (x.date_key==-1).any(); display(x.head()); print("checks: OK")